In [0]:
from pyspark.sql.functions import (
    col, trim, lower, to_date, current_timestamp, coalesce, lit,
    row_number, year, month, dayofmonth, date_trunc, min as min_, max as max_
)
from pyspark.sql.window import Window

silver_customers = "novocart_global.novocart_silver.customers_cleaned"
silver_products = "novocart_global.novocart_silver.products_cleaned"
silver_orders = "novocart_global.novocart_silver.orders_cleaned"
silver_order_items = "novocart_global.novocart_silver.order_items_cleaned"
silver_exchange = "novocart_global.novocart_silver.exchange_rates_cleaned"

gold_dim_customer = "novocart_global.novocart_gold.dim_customer"
gold_dim_product = "novocart_global.novocart_gold.dim_product"
gold_dim_date = "novocart_global.novocart_gold.dim_date"
gold_dim_channel = "novocart_global.novocart_gold.dim_channel"
gold_dim_country = "novocart_global.novocart_gold.dim_country"
gold_fact_sales = "novocart_global.novocart_gold.fact_sales"

## DIM: CUSTOMER

In [0]:
cust_df = spark.table(silver_customers).select(
    col("customer_id"),
    trim(col("customer_name")).alias("customer_name"),
    lower(trim(col("email"))).alias("email"),
    to_date(col("registration_date")).alias("registration_date"),
    trim(col("country_code")).alias("country_code"),
    trim(col("channel")).alias("channel")
).filter(col("customer_id").isNotNull())

w_cust = Window.orderBy("customer_id")
dim_customer = cust_df.withColumn("customer_key", row_number().over(w_cust)).select(
    "customer_key", "customer_id", "customer_name", "email", "country_code", "channel", "registration_date"
)

dim_customer.write.format("delta").mode("overwrite").saveAsTable(gold_dim_customer)

## DIM: Products

In [0]:

prod_df = spark.table(silver_products).select(
    col("product_id"),
    trim(col("product_name")).alias("product_name"),
    trim(col("category")).alias("category"),
    col("price").cast("double").alias("price"),
    trim(col("currency")).alias("currency"),
    trim(col("country_code")).alias("country_code")
).filter(col("product_id").isNotNull())

w_prod = Window.orderBy("product_id")
dim_product = prod_df.withColumn("product_key", row_number().over(w_prod)).select(
    "product_key", "product_id", "product_name", "category", "price", "currency", "country_code"
)

dim_product.write.format("delta").mode("overwrite").saveAsTable(gold_dim_product)


## DIM: Date


In [0]:

orders_df = spark.table(silver_orders).select(to_date(col("order_date")).alias("order_date")).filter(col("order_date").isNotNull())
min_max = orders_df.agg(min_("order_date").alias("min_date"), max_("order_date").alias("max_date")).collect()[0]
min_date = min_max["min_date"]
max_date = min_max["max_date"]

# build calendar
from pyspark.sql.functions import sequence, explode, expr
calendar = spark.range(1).select(
    explode(sequence(expr(f"date('{min_date}')"), expr(f"date('{max_date}')"))).alias("dt")
).select(
    col("dt").alias("date"),
    year(col("dt")).alias("year"),
    month(col("dt")).alias("month"),
    dayofmonth(col("dt")).alias("day"),
    date_trunc("month", col("dt")).alias("month_start")
)
w_date = Window.orderBy("date")
dim_date = calendar.withColumn("date_key", row_number().over(w_date)).select("date_key", "date", "year", "month", "day", "month_start")
dim_date.write.format("delta").mode("overwrite").saveAsTable(gold_dim_date)


## DIM: Channel

In [0]:

channels = spark.table(silver_orders).select(trim(col("channel")).alias("channel")).distinct().filter(col("channel").isNotNull())
w_ch = Window.orderBy("channel")
dim_channel = channels.withColumn("channel_key", row_number().over(w_ch)).select("channel_key", "channel")
dim_channel.write.format("delta").mode("overwrite").saveAsTable(gold_dim_channel)

## DIM: Country

In [0]:

countries = spark.table(silver_customers).select(trim(col("country_code")).alias("country_code")).distinct().filter(col("country_code").isNotNull())
w_ct = Window.orderBy("country_code")
dim_country = countries.withColumn("country_key", row_number().over(w_ct)).select("country_key", "country_code")
dim_country.write.format("delta").mode("overwrite").saveAsTable(gold_dim_country)


## FACT: Sales

In [0]:

orders = spark.table(silver_orders).select(
    col("order_id"),
    col("customer_id"),
    to_date(col("order_date")).alias("order_date"),
    trim(col("channel")).alias("channel"),
    trim(col("currency")).alias("currency_code"),
    col("order_status"),
    trim(col("country_code")).alias("country_code")
)

order_items = spark.table(silver_order_items).select(
    col("order_item_id"),
    col("order_id"),
    col("product_id"),
    col("quantity").cast("long").alias("quantity"),
    col("unit_price").cast("double").alias("unit_price"),
    col("line_total").cast("double").alias("line_total")
).filter((col("quantity") > 0) & (col("unit_price") >= 0))

# exchange rates:
exchange = spark.table(silver_exchange).select(
    trim(col("currency_code")).alias("currency_code"),
    col("exchange_rate_to_usd").cast("double").alias("exchange_rate_to_usd")
)

# join orders -> items (item-granular) and enrich with dims
fact = order_items.join(orders, "order_id", "left")

# join exchange rate (use rate 1 for missing)
fact = fact.join(exchange, fact["currency_code"] == exchange["currency_code"], "left") \
           .withColumn("exchange_rate_to_usd", coalesce(col("exchange_rate_to_usd"), lit(1.0)))

# compute USD amounts
fact = fact.withColumn("line_total_usd", (col("line_total") * col("exchange_rate_to_usd")).cast("double")) \
           .withColumn("unit_price_usd", (col("unit_price") * col("exchange_rate_to_usd")).cast("double"))

# attach surrogate keys from dims (left joins; dims have unique keys)
fact = fact.join(dim_customer.select("customer_key", "customer_id"), "customer_id", "left") \
           .join(dim_product.select("product_key", "product_id"), "product_id", "left") \
           .join(dim_date.select("date_key", "date"), fact["order_date"] == dim_date["date"], "left") \
           .join(dim_channel.select("channel_key", "channel"), "channel", "left") \
           .join(dim_country.select("country_key", "country_code"), "country_code", "left")

# final select with validations
fact_sales = fact.select(
    col("order_id"),
    col("order_item_id"),
    col("customer_key"),
    col("product_key"),
    col("date_key"),
    col("channel_key"),
    col("country_key"),
    col("quantity"),
    col("unit_price"),
    col("unit_price_usd"),
    col("line_total"),
    col("line_total_usd"),
    col("order_status"),
    col("order_date")
)

# persist fact_sales partitioned by month for performance
fact_sales = fact_sales.withColumn("order_month", date_trunc("month", col("order_date")))
fact_sales.write.format("delta").mode("overwrite").partitionBy("order_month").saveAsTable(gold_fact_sales)